# 02_logic_constraint_prototype.ipynb
Prototyping the Logic-Constraint Engine using OR-Tools.

In [13]:
from google.colab import drive
drive.mount("/content/drive")

ROOT_PAHT = "/content/drive/MyDrive/projects/DLThon2"
#ROOT_PAHT = ".."

os.chdir(ROOT_PAHT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
import sys
from ortools.sat.python import cp_model

from src.constraint_solver import NarrativeConstraintSolver

### 1. `NarrativeConstraintSolver` 활용 시퀀스 생성
이미 정의된 `NarrativeConstraintSolver`를 사용하여 이론적으로 타당한 내러티브 시퀀스를 생성합니다.

In [10]:
# 프롭(Propp) 이론 기반 5단계 시퀀스 생성
propp_solver = NarrativeConstraintSolver(theory_type="propp")
sequence = propp_solver.solve_sequence(length=5)

print("Generated Propp Sequence:")
for i, node_id in enumerate(sequence):
    node_info = next(n for n in propp_solver.nodes if n['id'] == node_id)
    print(f"Step {i+1}: {node_id} ({node_info['name']})")

Generated Propp Sequence:
Step 1: P01 (Initial Situation)
Step 2: P02 (Absentiation)
Step 3: P03 (Interdiction)
Step 4: P04 (Violation)
Step 5: P05 (Reconnaissance)


### 2. CP-SAT 모델 직접 구축 및 커스텀 제약 조건
기본 엔진을 넘어, 특정 사건을 반드시 포함하도록 하거나 특정 흐름을 금지하는 등의 커스텀 제약 조건을 추가하는 방법입니다.

In [11]:
def solve_with_custom_constraints(theory_type="propp", length=5, must_include=None):
    solver_engine = NarrativeConstraintSolver(theory_type=theory_type)
    model = cp_model.CpModel()
    num_nodes = len(solver_engine.nodes)

    # 변수 정의: 각 단계별 노드 인덱스
    steps = [model.NewIntVar(0, num_nodes - 1, f'step_{i}') for i in range(length)]

    # 1. 시나리오 시작 조건 (예: P01로 시작)
    model.Add(steps[0] == solver_engine.id_to_index["P01"])

    # 2. 전이 규칙 (Transition Rules)
    for i in range(length - 1):
        for curr_idx in range(num_nodes):
            curr_id = solver_engine.index_to_id[curr_idx]
            valid_next = solver_engine.get_valid_next_ids(curr_id)
            valid_indices = [solver_engine.id_to_index[nid] for nid in valid_next if nid in solver_engine.id_to_index]

            # curr_idx가 현재 단계(i)에 선택되었다면, 다음 단계(i+1)는 valid_indices 중 하나여야 함
            at_curr = model.NewBoolVar(f'at_{i}_{curr_idx}')
            model.Add(steps[i] == curr_idx).OnlyEnforceIf(at_curr)
            model.Add(steps[i] != curr_idx).OnlyEnforceIf(at_curr.Not())

            if not valid_indices:
                model.Add(at_curr == 0)
            else:
                model.AddAllowedAssignments([steps[i+1]], [(idx,) for idx in valid_indices]).OnlyEnforceIf(at_curr)

    # 3. 커스텀 제약: 특정 노드(must_include)가 시퀀스 어딘가에 반드시 포함되어야 함
    if must_include and must_include in solver_engine.id_to_index:
        target_idx = solver_engine.id_to_index[must_include]
        presence_bools = [model.NewBoolVar(f'present_{must_include}_at_{i}') for i in range(length)]
        for i in range(length):
            model.Add(steps[i] == target_idx).OnlyEnforceIf(presence_bools[i])
            model.Add(steps[i] != target_idx).OnlyEnforceIf(presence_bools[i].Not())
        model.Add(sum(presence_bools) >= 1)

    # 해결
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        return [solver_engine.index_to_id[solver.Value(s)] for s in steps]
    return []

# 'P04(Violation)'가 반드시 포함되는 6단계 시퀀스 생성
custom_seq = solve_with_custom_constraints(length=6, must_include="P04")
print("Custom Sequence (must include P04):", custom_seq)

Custom Sequence (must include P04): []


### 3. 보글러(Vogler) 이론을 이용한 해결
동일한 로직을 보글러의 12단계 이론에 적용해 봅니다.

In [12]:
vogler_solver = NarrativeConstraintSolver(theory_type="vogler")
v_sequence = vogler_solver.solve_sequence(length=3)
print("Vogler Journey Sequence:", v_sequence)

Vogler Journey Sequence: ['V01', 'V02', 'V03']
